# 02 — Matched location pricing

Same-job, same-car **airport versus downtown** price comparison for the three
approved city pairs. This notebook only orchestrates and presents: the
validated pricing pipeline, matching, summaries, statistical tests and the
figure all come from `ql2_sixt_canada_analysis.matched_location_pricing`.

It does not depend on notebook 01 or on any earlier kernel state. It rebuilds
every foundational gate from the raw files through the package
(`run_matched_location_pricing` → `run_pricing_pipeline`) and stops - showing
only the blocker categories - unless pricing readiness is true.

**Grain.** One matched pair is one approved city, one shared trusted
collection job (proven from the authority-backed schedule and the derived job
linkage key), one rental period, one exact approved vehicle product, one
currency and one price basis, with exactly one airport offer and exactly one
canonical downtown offer. Vancouver `Thurlow` is a governed alias combined
into canonical `Vancouver Downtown` before matching. Premiums are airport
minus downtown: positive is an airport premium, negative an airport discount.

## Confidentiality

All QL2 data is proprietary. This notebook displays aggregate counts,
statistics and the main figure only - never frames, rows, identifiers,
timestamps, file names or paths. Clear all outputs before committing.

In [ ]:
import io

from IPython.display import Image, display

from ql2_sixt_canada_analysis import (
    city_summary_frame,
    match_count_frame,
    plot_matched_location_premiums,
    run_matched_location_pricing,
    vehicle_type_summary_frame,
    vehicle_type_test_frame,
    write_matched_location_pricing_deliverables,
)
from ql2_sixt_canada_analysis.paths import resolve_raw_data_dir

# Raw-data directory: the centralized default unless automated validation sets the
# documented environment override. Set RAW_DIR_OVERRIDE only for a deliberate local
# experiment; never commit a value.
RAW_DIR_OVERRIDE = None
# Optional output directory for the Git-ignored report and figure. None writes nothing;
# use the documented command-line entry point to generate the local deliverables.
REPORTS_DIR = None

raw_dir = resolve_raw_data_dir(RAW_DIR_OVERRIDE)
print("Raw-data directory selected.")

In [ ]:
# Every readiness gate, the frame bindings, location authority, canonical offers, schedule
# evidence and vehicle stability are checked inside the package before any price is compared.
result = run_matched_location_pricing(raw_dir)
report = result.report
print("Matched location pricing status:", report.status.value)
if report.completed:
    print("Approved location pairs (canonical keys):")
    for airport, downtown in report.approved_pairs:
        print(f"  {' / '.join(airport)} versus {' / '.join(downtown)}")
else:
    print("Blocked - no commercial result is calculated.")
    print("Blockers:", ", ".join(b.value for b in report.blockers))
    if report.readiness_blockers:
        print("Pricing-readiness blockers:", ", ".join(report.readiness_blockers))

**Interpretation.** A `completed` status means every readiness gate, frame binding and same-job proof passed, and the approved pairs listed are the only comparisons made. A `blocked` status names the failing categories, and no premium below may be read.

## Match counts and attrition

Candidate match groups are identities with an offer on at least one side of
an approved pair. Match rate = matched pairs / candidate groups. One-sided
groups are unmatched; several price-distinct offers on one side are
ambiguous; currency or basis disagreements are incompatible.

In [ ]:
if report.completed:
    display(match_count_frame(result))
else:
    print("Skipped: matched location pricing is blocked.")

**Interpretation.** Matched pairs are the denominator of every premium below. A low match rate or many one-sided groups means the premiums describe only products both locations returned in the same job; ambiguous and incompatible groups are excluded, not imputed.

## Premium distribution by city

Dollar and percentage premiums with mean, median, dispersion, quartiles,
range and the share of pairs where the airport is higher, equal or lower.

In [ ]:
if report.completed:
    display(city_summary_frame(result))
else:
    print("Skipped: matched location pricing is blocked.")

**Interpretation.** The median premium and the airport-higher share are the headline: a positive median means the airport listed a higher price for the same product in the same job. Dispersion and range show whether that holds broadly or depends on a few products; percentages exclude zero downtown prices.

## Main figure

Distribution of matched premiums by city: boxes show the interquartile range
and median, whiskers the full range, points every matched pair; the dashed
line is zero premium.

In [ ]:
if report.completed:
    figure, _ = plot_matched_location_premiums(result)
    buffer = io.BytesIO()
    figure.savefig(buffer, format="png", dpi=figure.dpi)
    display(Image(data=buffer.getvalue()))
else:
    print("Skipped: matched location pricing is blocked.")

**Interpretation.** A box whose interquartile range lies entirely on one side of zero shows that the middle half of that city's observed matched-pair premiums has the same sign; individual pairs may still differ. Points on both sides of zero show that the observed premium sign varies across matched pairs; they do not show that product identity causes or explains that variation. Capture timing, rental context and repeated measurements of the same products remain unresolved contributors. The figure contains matched pairs only and supports a descriptive, associational reading.

## Vehicle type

Descriptive premiums by city and `car_type`, then Kruskal-Wallis tests of the
premium distributions across vehicle types within each city (minimum
observations per type fixed in the package; Holm adjustment across cities;
epsilon-squared effect size; `not_testable` when the sample rule fails).
These are associational comparisons of observed offers, not causal claims,
and hourly captures of one product are repeated measurements.

In [ ]:
if report.completed:
    display(vehicle_type_summary_frame(result))
    display(vehicle_type_test_frame(result))
else:
    print("Skipped: matched location pricing is blocked.")

**Interpretation.** Vehicle-type differences are associational: a small adjusted p-value with a meaningful epsilon-squared suggests premiums differ by vehicle class within a city, but repeated hourly captures of the same products limit independence. `not_testable` means too few observations, not equal premiums.

## Limitations

- Premiums compare listed prices of identical products in the same collection
  job; they are not transacted prices, total customer cost or causal effects.
- The sample spans roughly 90 hours of hourly captures, so it cannot show
  seasonality or whether the premiums persist.
- Unmatched, ambiguous and incompatible groups are excluded, so the premiums
  describe the products both locations returned.
- Capture-time (hour-of-day) and price-percentile views of the premium are
  planned extensions and are not presented here.

## Data-plan reconciliation

- City: presented as match counts, distribution summaries and the main figure per approved city pair.
- Vehicle type: presented as descriptive summaries and Kruskal-Wallis tests within each city (associational).
- Capture time: pairs are matched within one scheduled capture; an hour-of-day breakdown is not presented (planned extension).
- Price percentile: premium quartiles and ranges are presented; a premium-by-price-level view is not presented (planned extension).
- Percentage of matches where the airport is higher: presented as the positive, zero and negative shares by city.

## Local report

The Git-ignored Markdown report and figure are written only when `REPORTS_DIR`
is set above; the documented command-line entry point generates them too.

In [ ]:
# Writes the Git-ignored report and figure only when REPORTS_DIR is set explicitly.
if report.completed and REPORTS_DIR is not None:
    from pathlib import Path

    write_matched_location_pricing_deliverables(
        result, report_path=Path(REPORTS_DIR) / "matched_location_pricing.md",
        figure_path=Path(REPORTS_DIR) / "figures" / "matched_location_pricing.png")
    print("Local report and figure written.")
else:
    print("No files written.")